# 05 — Retrieval

Compare four retriever strategies against the same FAISS index.

| Strategy | Key parameter | Use case |
|---|---|---|
| Similarity | `k` | Standard top-K search |
| MMR | `k`, `fetch_k`, `lambda_mult` | Diverse results |
| Score threshold | `k`, `score_threshold` | Quality over quantity |
| Metadata filter | `filter` | Restrict by metadata |

**Config source:** `configs/default.yaml` → `paths.faiss_index`, `notebooks.retrieval_queries`

In [ ]:
import sys
import os
sys.path.append("..")

from rag_pipeline.utils import load_notebook_config
from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.vectorstores import load_vectorstore
from rag_pipeline.retrieval import build_retriever

cfg, REPO = load_notebook_config()
FAISS_DIR = REPO / cfg.paths["faiss_index"]
QUERIES   = cfg.notebooks["retrieval_queries"]

print("FAISS index dir:", FAISS_DIR)
print("Queries loaded: ", len(QUERIES))

**Load vector store**

In [ ]:
emb = build_embeddings(dict(cfg.embeddings))
store = load_vectorstore(emb, {
    "type": "faiss",
    "persist_dir": str(FAISS_DIR),
})
print("Index size:", store.index.ntotal)

**Similarity search**

In [ ]:
r = build_retriever(store, {"search_type": "similarity", "k": 3})
for q in QUERIES:
    print(f"\nQ: {q}")
    for d in r.invoke(q):
        print(f"  row={d.metadata.get('row')} | {d.page_content[:80]}...")

**MMR**

In [ ]:
r_mmr = build_retriever(store, {
    "search_type": "mmr", "k": 3, "fetch_k": 20, "lambda_mult": 0.5,
})
print("MMR top-3 for first query:")
for d in r_mmr.invoke(QUERIES[0]):
    print(f"  row={d.metadata.get('row')} | {d.page_content[:80]}...")

**Score threshold**

In [ ]:
r_thr = build_retriever(store, {
    "search_type": "similarity_score_threshold",
    "k": 5,
    "score_threshold": 0.6,
})
print("Threshold top-5 for first query:")
for d in r_thr.invoke(QUERIES[0]):
    print(f"  row={d.metadata.get('row')} | {d.page_content[:80]}...")

**Metadata filter**

In [ ]:
r_fil = build_retriever(store, {
    "search_type": "similarity", "k": 3, "filter": {"row": 2},
})
print("Filtered (row=2) top-3:")
for d in r_fil.invoke(QUERIES[0]):
    print(f"  row={d.metadata.get('row')} | {d.page_content[:80]}...")

In [ ]:
CONFIGS = {
    "similarity k=3":    {"search_type": "similarity", "k": 3},
    "mmr k=3 λ=0.5":     {"search_type": "mmr", "k": 3, "fetch_k": 20, "lambda_mult": 0.5},
    "threshold 0.6":     {"search_type": "similarity_score_threshold", "k": 5, "score_threshold": 0.6},
    "filter row=2":      {"search_type": "similarity", "k": 3, "filter": {"row": 2}},
}

q = QUERIES[0]
print(f"Query: {q}\n{'='*80}")
for name, rcfg in CONFIGS.items():
    rows = [d.metadata.get("row") for d in build_retriever(store, rcfg).invoke(q)]
    print(f"{name:<20} → rows {rows}")